# Monthly multiplicative QDM grid-cell bias correction (2015–2100)

This notebook is a **QDM replacement** for your previous Hempel/ISI-MIP grid-calibration notebook.

It keeps the same data layout:

- model input: yearly aggregated multi-band GeoTIFF  
  `prediction/{product}_ssp_daily_tif_yearly/{ssp}/{model}/{year}.tif`
- observation input: `calibration/hist_{product}.nc`
- default output: one corrected multi-band GeoTIFF per `(product, ssp, model, year)`

## QDM used here

For each grid cell and calendar month:

\[
\tau = F_{m,p}(x_{m,p})
\]

\[
\Delta = \frac{x_{m,p}}{F^{-1}_{m,c}(\tau)}
\]

\[
\hat{x}_{m,p}=F^{-1}_{o,c}(\tau)\Delta
\]

This is **multiplicative / ratio Quantile Delta Mapping**, used to preserve relative changes in quantiles.

### Implementation choices

1. Monthly calibration distributions.
2. Ratio QDM for non-negative concentrations.
3. Near-zero left-censor treatment using `TRACE_CALC`, with output values below `TRACE` set to zero.
4. `RATIO_MAX` caps unstable ratios when the calibration-model quantile is near zero.
5. Future `F_m,p` is estimated with a moving **9-year** window by default.
6. Years before `QDM_PROJECTION_START` use the current-period empirical quantile mapping part of QDM.
7. Negative finite input values are clipped to zero before ratio-QDM, and final finite output is constrained to `>= 0`.

> If `fireo3` is scientifically defined as `fire-on minus fire-off` and negative values are meaningful, ratio/non-negative QDM is not appropriate for that variable; use additive QDM instead.

Before running the full 17 GCM × 4 SSP × 2 pollutant job, run one model/SSP/product smoke test and inspect the annual trend and high quantiles.

## References

- Cannon, A.J., Sobie, S.R., & Murdock, T.Q. (2015). *Bias correction of simulated precipitation by quantile mapping: How well do methods preserve relative changes in quantiles and extremes?* Journal of Climate, 28, 6938–6959. DOI: 10.1175/JCLI-D-14-00754.1
- The near-zero `trace`, `ratio.max`, and multiplicative-delta logic follows the design of `MBC::QDM`.


In [ ]:
import datetime as dt
import gc
import os
import json as jsonlib
import sys
import traceback
import zlib
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import numpy as np

if not sys.platform.startswith("linux"):
    raise RuntimeError("This notebook is written for a Linux kernel")

In [ ]:
# ================================================================
# USER CONFIGURATION
# ================================================================

PROJECT_DIR = Path(os.environ.get("AU_FIRE_ROOT", str(Path.cwd().parent / "data")))

MODELS = [
    "ACCESS-CM2",
    "ACCESS-ESM1-5",
    "BCC-CSM2-MR",
    "CanESM5",
    "CMCC-ESM2",
    "EC-Earth3",
    "EC-Earth3-Veg-LR",
    "GFDL-ESM4",
    "INM-CM4-8",
    "INM-CM5-0",
    "KACE-1-0-G",
    "MPI-ESM1-2-HR",
    "MPI-ESM1-2-LR",
    "MRI-ESM2-0",
    "NorESM2-LM",
    "NorESM2-MM",
    "TaiESM1",
]

SSPS = ["ssp126", "ssp245", "ssp370", "ssp585"]

PRODUCTS = [
    "firepm25",
    "fireo3",
]

# Calibration overlap
FIT_START = dt.date(2015, 1, 1)
FIT_END = dt.date(2020, 12, 31)

# Output period
APPLY_START = dt.date(2015, 1, 1)
APPLY_END = dt.date(2100, 12, 31)

# Future QDM starts after calibration by default.
# If your final future period starts in 2021, the cleanest setup is usually:
# FIT_END = dt.date(2020, 12, 31)
# QDM_PROJECTION_START = dt.date(2021, 1, 1)
QDM_PROJECTION_START = FIT_END + dt.timedelta(days=1)

SSP_YEARLY_ROOT_REL = "prediction/{product}_ssp_daily_tif_yearly"
OBS_NC_REL = "calibration/hist_{product}.nc"
OBS_VARIABLE = "hist_{product}"

PARAMS_DIR = PROJECT_DIR / "prediction/qdm_grid_calibration_parameters_2015_2020"

OUTPUT_MODE = "yearly"
OUTPUT_DIR = PROJECT_DIR / (
    "prediction/qdm_grid_calibrated_yearly_tif_2015_2100"
    if OUTPUT_MODE == "yearly"
    else "prediction/qdm_grid_calibrated_daily_tif_2015_2100"
)

# ---------------- QDM settings ----------------

# Number of stored empirical quantile nodes per month/cell.
N_QUANTILES = 41
TAU_MIN = 0.001
TAU_MAX = 0.999

# Moving future window used to estimate F_m,p.
# 9 years gives a monthly sample size close to your 2015-2023 calibration.
QDM_WINDOW_YEARS = 9

# Near-zero handling; tune after checking the units/distributions.
TRACE_BY_PRODUCT = {
    "firepm25": 0.01,
    "fireo3": 0.01,
}

TRACE_CALC_BY_PRODUCT = {
    "firepm25": 0.005,
    "fireo3": 0.005,
}

# Mirrors the ratio-stabilisation idea used by MBC::QDM.
RATIO_MAX = 2.0
RATIO_MAX_TRACE_FACTOR = 10.0

MIN_CAL_DAYS_PER_MONTH = 30

# Reduce RAM used when inverting quantile curves.
DAY_CHUNK = 8

# Reproducible left-censor jitter.
RANDOM_SEED = 20260814

# QDM is much heavier than the old Hempel arithmetic.
# Start conservatively and increase only after checking RAM and S-drive I/O.
FIT_PROCESSES = 1
APPLY_PROCESSES = 4

SKIP_EXISTING = True

for path in (PARAMS_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_DIR          = {PROJECT_DIR}")
print(f"Fit window           = {FIT_START} .. {FIT_END}")
print(f"Apply window         = {APPLY_START} .. {APPLY_END}")
print(f"QDM projection start = {QDM_PROJECTION_START}")
print(f"QDM window           = {QDM_WINDOW_YEARS} years")
print(f"Quantile nodes       = {N_QUANTILES}")
print(f"Products             = {PRODUCTS}")
print(f"Models               = {len(MODELS)}")
print(f"SSPs                 = {SSPS}")
print(f"PARAMS_DIR           = {PARAMS_DIR}")
print(f"OUTPUT_DIR           = {OUTPUT_DIR}")

## QDM core

The fitting stage stores monthly empirical observation/model quantile curves for each grid cell.  
The apply stage uses those curves with either current-period empirical quantile mapping or future ratio-QDM.

In [ ]:
# --------------------------------------------------------------------
# date / grouping helpers
# --------------------------------------------------------------------

def date_range(start, end):
    """Inclusive list of consecutive dates."""
    return [start + dt.timedelta(days=i) for i in range((end - start).days + 1)]


def month_index(dates):
    """0..11 month index for each date."""
    return np.fromiter((d.month - 1 for d in dates), dtype=np.int8, count=len(dates))


def stable_seed(*parts):
    """Stable uint32 seed independent of Python's randomized hash()."""
    text = "|".join(str(x) for x in parts).encode("utf-8")
    return (zlib.crc32(text) + int(RANDOM_SEED)) & 0xFFFFFFFF


def projection_window_years(target_year, first_year, last_year, width):
    """Fixed-width centered year window, shifted at boundaries."""
    width = int(width)
    if width < 1:
        raise ValueError("width must be >= 1")

    total = last_year - first_year + 1
    if total <= width:
        return list(range(first_year, last_year + 1))

    half = width // 2
    start = target_year - half
    end = start + width - 1

    if start < first_year:
        start = first_year
        end = start + width - 1

    if end > last_year:
        end = last_year
        start = end - width + 1

    return list(range(start, end + 1))

In [ ]:
# --------------------------------------------------------------------
# QDM numerical helpers
# --------------------------------------------------------------------

def _clip_and_censor(values, trace_calc, rng):
    """
    Ratio-QDM preprocessing:
      - preserve NaN
      - finite negatives -> 0
      - finite values below trace_calc -> Uniform(eps, trace_calc)
    """
    x = np.asarray(values, dtype=np.float64).copy()
    finite = np.isfinite(x)
    x[finite] = np.maximum(x[finite], 0.0)

    if trace_calc > 0:
        censored = finite & (x < trace_calc)
        n = int(censored.sum())
        if n:
            eps = np.finfo(np.float64).eps
            x[censored] = rng.uniform(eps, trace_calc, size=n)

    return x


def _quantile_curve_at_tau(qcurve, tau_values, taus):
    """
    Interpolate a per-cell quantile curve at arbitrary tau.

    qcurve     : (nq, cells)
    tau_values : (time, cells) or (cells,)
    """
    qcurve = np.asarray(qcurve, dtype=np.float64)
    tau = np.asarray(tau_values, dtype=np.float64)

    squeeze = tau.ndim == 1
    if squeeze:
        tau = tau[None, :]

    nq, cells = qcurve.shape
    if tau.shape[1] != cells:
        raise ValueError("tau_values and qcurve cell dimensions differ")

    tau_clip = np.clip(tau, taus[0], taus[-1])

    step = (taus[-1] - taus[0]) / (nq - 1)
    u = (tau_clip - taus[0]) / step

    i0 = np.floor(u).astype(np.int64)
    i0 = np.clip(i0, 0, nq - 2)
    i1 = i0 + 1
    frac = u - i0

    cols = np.arange(cells)[None, :]
    q0 = qcurve[i0, cols]
    q1 = qcurve[i1, cols]

    out = q0 + frac * (q1 - q0)

    invalid = (
        ~np.isfinite(tau)
        | ~np.isfinite(qcurve[0])[None, :]
        | ~np.isfinite(qcurve[-1])[None, :]
    )
    out[invalid] = np.nan

    return out[0] if squeeze else out


def _values_to_tau(values, qcurve, taus, day_chunk=8):
    """
    Approximate F(x) by inverting a monotone quantile curve.

    values : (time, cells)
    qcurve : (nq, cells)
    """
    values = np.asarray(values, dtype=np.float64)
    qcurve = np.asarray(qcurve, dtype=np.float64)

    nt, cells = values.shape
    nq = qcurve.shape[0]

    if qcurve.shape[1] != cells:
        raise ValueError("values and qcurve cell dimensions differ")

    out = np.full((nt, cells), np.nan, dtype=np.float64)
    cols = np.arange(cells)[None, :]

    for start in range(0, nt, day_chunk):
        stop = min(nt, start + day_chunk)
        v = values[start:stop]

        count_le = (v[:, None, :] >= qcurve[None, :, :]).sum(axis=1)

        i0 = np.clip(count_le - 1, 0, nq - 2).astype(np.int64)
        i1 = i0 + 1

        q0 = qcurve[i0, cols]
        q1 = qcurve[i1, cols]

        t0 = taus[i0]
        t1 = taus[i1]

        denom = q1 - q0

        frac = np.zeros_like(v, dtype=np.float64)
        regular = np.isfinite(denom) & (denom > 0)

        np.divide(
            v - q0,
            denom,
            out=frac,
            where=regular,
        )

        frac = np.clip(frac, 0.0, 1.0)

        # Flat plateau / tie.
        flat = np.isfinite(denom) & (denom <= 0)
        frac[flat & (v >= q1)] = 1.0

        tau = t0 + frac * (t1 - t0)

        tau = np.where(
            v <= qcurve[0][None, :],
            taus[0],
            tau,
        )

        tau = np.where(
            v >= qcurve[-1][None, :],
            taus[-1],
            tau,
        )

        invalid = (
            ~np.isfinite(v)
            | ~np.isfinite(qcurve[0])[None, :]
            | ~np.isfinite(qcurve[-1])[None, :]
        )

        tau[invalid] = np.nan
        out[start:stop] = tau

    return out

In [ ]:
# --------------------------------------------------------------------
# Fit monthly calibration quantiles
# --------------------------------------------------------------------

def fit_qdm_quantiles(
    obs,
    mod,
    dates,
    taus,
    trace_calc,
    min_cal_days=30,
    seed_prefix="fit",
):
    """
    Fit monthly empirical quantile curves for each grid cell.

    Returns
    -------
    obs_q, mod_q : (12, nq, H, W), float32

    Only overlapping complete-case days are retained for each cell.
    """
    if obs.shape != mod.shape:
        raise ValueError(f"obs shape {obs.shape} != mod shape {mod.shape}")

    time_n, height, width = obs.shape
    cells = height * width
    months = month_index(dates)
    nq = len(taus)

    obs_q = np.full((12, nq, cells), np.nan, dtype=np.float32)
    mod_q = np.full((12, nq, cells), np.nan, dtype=np.float32)

    obs2 = obs.reshape(time_n, cells)
    mod2 = mod.reshape(time_n, cells)

    for m in range(12):
        sel = np.flatnonzero(months == m)

        o = obs2[sel].astype(np.float64, copy=True)
        mc = mod2[sel].astype(np.float64, copy=True)

        joint = np.isfinite(o) & np.isfinite(mc)
        o[~joint] = np.nan
        mc[~joint] = np.nan

        valid_count = joint.sum(axis=0)
        idx = np.flatnonzero(valid_count >= min_cal_days)

        if idx.size == 0:
            continue

        rng_o = np.random.default_rng(
            stable_seed(seed_prefix, m, "obs")
        )
        rng_m = np.random.default_rng(
            stable_seed(seed_prefix, m, "mod")
        )

        o = _clip_and_censor(o, trace_calc, rng_o)
        mc = _clip_and_censor(mc, trace_calc, rng_m)

        oq = np.nanquantile(
            o[:, idx],
            taus,
            axis=0,
        )

        mq = np.nanquantile(
            mc[:, idx],
            taus,
            axis=0,
        )

        # Quantile curves must be non-decreasing.
        oq = np.maximum.accumulate(oq, axis=0)
        mq = np.maximum.accumulate(mq, axis=0)

        obs_q[m][:, idx] = oq.astype(np.float32)
        mod_q[m][:, idx] = mq.astype(np.float32)

    return (
        obs_q.reshape(12, nq, height, width),
        mod_q.reshape(12, nq, height, width),
    )

In [ ]:
# --------------------------------------------------------------------
# Apply current-period mapping
# --------------------------------------------------------------------

def apply_current_qm(
    raw,
    dates,
    obs_q,
    mod_q,
    taus,
    trace,
    trace_calc,
    seed_prefix,
    day_chunk=8,
):
    """
    Current-period mapping:
      mhat.c = F_obs,c^-1(F_mod,c(m.c))
    """
    time_n, height, width = raw.shape
    cells = height * width

    raw2 = raw.reshape(time_n, cells)
    out = np.full((time_n, cells), np.nan, dtype=np.float64)

    months = month_index(dates)

    oq = obs_q.reshape(12, len(taus), cells)
    mq = mod_q.reshape(12, len(taus), cells)

    for m in range(12):
        idx_days = np.flatnonzero(months == m)

        if idx_days.size == 0:
            continue

        rng = np.random.default_rng(
            stable_seed(seed_prefix, "current", m)
        )

        x = _clip_and_censor(
            raw2[idx_days],
            trace_calc,
            rng,
        )

        tau = _values_to_tau(
            x,
            mq[m],
            taus,
            day_chunk=day_chunk,
        )

        corrected = _quantile_curve_at_tau(
            oq[m],
            tau,
            taus,
        )

        corrected[corrected < trace] = 0.0
        corrected = np.maximum(corrected, 0.0)

        corrected[
            ~np.isfinite(raw2[idx_days])
        ] = np.nan

        out[idx_days] = corrected

    return out.reshape(
        time_n,
        height,
        width,
    ).astype(np.float32)

In [ ]:
# --------------------------------------------------------------------
# Apply future multiplicative QDM
# --------------------------------------------------------------------

def apply_future_qdm_year(
    target_year,
    target_dates,
    target_raw,
    window_year_data,
    obs_q,
    mod_q,
    taus,
    trace,
    trace_calc,
    ratio_max,
    ratio_max_trace,
    seed_prefix,
    day_chunk=8,
):
    """
    Apply multiplicative QDM to one target year.

    window_year_data:
        dict year -> (dates, raw yearly array)

    F_m,p is estimated separately for each calendar month using all
    same-month values in the moving projection window.
    """
    time_n, height, width = target_raw.shape
    cells = height * width

    out = np.full(
        (time_n, cells),
        np.nan,
        dtype=np.float64,
    )

    target_months = month_index(target_dates)

    oq = obs_q.reshape(
        12,
        len(taus),
        cells,
    )

    mq = mod_q.reshape(
        12,
        len(taus),
        cells,
    )

    for m in range(12):
        target_idx = np.flatnonzero(
            target_months == m
        )

        if target_idx.size == 0:
            continue

        pieces = []
        piece_dates = []

        for yy in sorted(window_year_data):
            yy_dates, yy_raw = window_year_data[yy]
            yy_months = month_index(yy_dates)

            ii = np.flatnonzero(
                yy_months == m
            )

            if ii.size:
                pieces.append(
                    yy_raw[ii]
                )

                piece_dates.extend(
                    [yy_dates[k] for k in ii]
                )

        if not pieces:
            continue

        proj = np.concatenate(
            pieces,
            axis=0,
        ).reshape(
            -1,
            cells,
        )

        piece_dates = np.asarray(
            piece_dates,
            dtype=object,
        )

        # m.p zero/near-zero treatment.
        rng = np.random.default_rng(
            stable_seed(
                seed_prefix,
                "future",
                target_year,
                m,
            )
        )

        proj_proc = _clip_and_censor(
            proj,
            trace_calc,
            rng,
        )

        target_date_set = set(target_dates)

        target_mask = np.fromiter(
            (d in target_date_set for d in piece_dates),
            dtype=bool,
            count=len(piece_dates),
        )

        x = proj_proc[
            target_mask
        ]

        if x.shape[0] != target_idx.size:
            raise ValueError(
                f"{target_year} month {m+1}: "
                f"target window has {x.shape[0]} days, "
                f"target array has {target_idx.size}"
            )

        # F_m,p^-1
        proj_q = np.nanquantile(
            proj_proc,
            taus,
            axis=0,
        )

        proj_q = np.maximum.accumulate(
            proj_q,
            axis=0,
        )

        # tau = F_m,p(x_m,p)
        tau = _values_to_tau(
            x,
            proj_q,
            taus,
            day_chunk=day_chunk,
        )

        # F_m,c^-1(tau)
        qmc = _quantile_curve_at_tau(
            mq[m],
            tau,
            taus,
        )

        # F_o,c^-1(tau)
        qoc = _quantile_curve_at_tau(
            oq[m],
            tau,
            taus,
        )

        with np.errstate(
            divide="ignore",
            invalid="ignore",
        ):
            delta = x / qmc

        eps = np.finfo(
            np.float64
        ).eps

        tiny = (
            ~np.isfinite(qmc)
            | (qmc <= eps)
        )

        delta = np.where(
            tiny & (x <= eps),
            1.0,
            delta,
        )

        delta = np.where(
            tiny & (x > eps),
            ratio_max,
            delta,
        )

        # Same stabilisation idea as MBC::QDM.
        cap = (
            np.isfinite(delta)
            & (delta > ratio_max)
            & np.isfinite(qmc)
            & (qmc < ratio_max_trace)
        )

        delta[cap] = ratio_max

        corrected = qoc * delta

        corrected[
            corrected < trace
        ] = 0.0

        corrected = np.maximum(
            corrected,
            0.0,
        )

        target_raw_month = target_raw[
            target_idx
        ].reshape(
            target_idx.size,
            cells,
        )

        corrected[
            ~np.isfinite(target_raw_month)
        ] = np.nan

        out[target_idx] = corrected

        del (
            proj,
            proj_proc,
            proj_q,
            x,
            tau,
            qmc,
            qoc,
            delta,
            corrected,
        )

        gc.collect()

    return out.reshape(
        time_n,
        height,
        width,
    ).astype(np.float32)

## Raster / NetCDF I/O

This is retained from your previous notebook so the QDM version reads/writes the same yearly multi-band TIFF format.

In [ ]:
# --------------------------------------------------------------------
# raster / NetCDF I/O (rasterio + netCDF4 imported lazily, inside these
# functions, so the fit/apply math above and the self-test below don't
# need them importable)
# --------------------------------------------------------------------

def read_year_multiband_tif(path):
    """
    Read one yearly-aggregated multi-band GeoTIFF (as produced by the
    aggregation notebook). Returns (dates, data, profile):
      dates   -- list of dt.date, one per band, in band order
      data    -- (n_bands, H, W) float32 array, NaN where missing
      profile -- rasterio profile dict (crs, transform, height, width, ...)
    Falls back to assuming band i = day i of the year encoded in the
    filename if the `band_dates` tag is absent (e.g. a hand-made file).
    """
    import rasterio

    with rasterio.open(path) as ds:
        data = ds.read().astype(np.float32)
        tags = ds.tags()
        if "band_dates" in tags:
            dates = [dt.date.fromisoformat(s) for s in jsonlib.loads(tags["band_dates"])]
        else:
            year = int(Path(path).stem)
            dates = date_range(dt.date(year, 1, 1), dt.date(year, 12, 31))
            if len(dates) != data.shape[0]:
                raise ValueError(
                    f"{path}: no band_dates tag and band count {data.shape[0]} != "
                    f"{len(dates)} days inferred from filename year {year}"
                )
        profile = ds.profile.copy()
    return dates, data, profile


def read_dates_from_yearly_dir(yearly_dir, dates):
    """
    Read an arbitrary chronological list of dates from a directory of
    per-year aggregated TIFFs ({year}.tif), as written by the aggregation
    notebook. Opens each needed year file once (not once per day) and
    transparently handles a date range spanning multiple years.

    A year file that is missing or fails to open leaves every requested
    date in that year as NaN *as long as at least one other requested
    year could be read* (so a genuinely partial gap doesn't abort an
    otherwise-good multi-year read). If EVERY touched year fails, there
    is no shape/profile information to build a fallback array from, so
    this raises FileNotFoundError instead of silently fabricating an
    all-NaN result -- for apply, an entire year of NaN "corrected" output
    would be a much worse silent failure than the task erroring out.

    Returns (stack, profile): stack is (len(dates), H, W) float32, aligned
    to the order of `dates`; profile is the rasterio profile from
    whichever year file was read first successfully.
    """
    by_year = defaultdict(list)
    for i, d in enumerate(dates):
        by_year[d.year].append((i, d))

    stack = None
    profile = None
    problem_years = []
    for year, items in by_year.items():
        year_path = yearly_dir / f"{year}.tif"
        try:
            year_dates, year_data, year_profile = read_year_multiband_tif(year_path)
        except Exception as error:
            problem_years.append((year, repr(error)))
            continue
        if profile is None:
            profile = year_profile
            height, width = year_data.shape[1:]
            stack = np.full((len(dates), height, width), np.nan, dtype=np.float32)
        date_to_band = {d: b for b, d in enumerate(year_dates)}
        for i, d in items:
            band = date_to_band.get(d)
            if band is not None:
                stack[i] = year_data[band]
            else:
                print(f"WARNING: {d} missing from {year_path} band_dates; left as NaN")

    if stack is None:
        raise FileNotFoundError(
            f"No yearly aggregated files could be read under {yearly_dir} for the "
            f"requested {len(dates)} date(s); years tried: {sorted(by_year.keys())}"
        )
    if problem_years:
        print(f"WARNING: {len(problem_years)} year file(s) missing/unreadable under {yearly_dir}: {problem_years}")
    return stack, profile


def read_obs_series(nc_path, variable, dates, shape):
    """
    Read the (time, H, W) slice of an observation NetCDF matching `dates`.
    Unchanged from before -- the observation file is already a single
    file covering 2000-2023, so there is nothing to aggregate there.

    Looks up each requested date against the file's own `time` coordinate
    (decoded via its units/calendar attributes) rather than assuming a
    fixed band offset, so it is safe to call with any sub-period of a
    longer observation record.
    """
    import netCDF4

    with netCDF4.Dataset(nc_path, "r") as dataset:
        time_var = dataset.variables.get("time")
        if time_var is None or "units" not in time_var.ncattrs():
            raise ValueError(
                f"{nc_path}: variable 'time' with a 'units' attribute is "
                "required to align dates"
            )
        calendar = getattr(time_var, "calendar", "standard")
        decoded = netCDF4.num2date(
            time_var[:], units=time_var.units, calendar=calendar,
            only_use_cftime_datetimes=False, only_use_python_datetimes=True,
        )
        time_index = {
            dt.date(int(t.year), int(t.month), int(t.day)): i
            for i, t in enumerate(decoded)
        }
        try:
            band_indices = [time_index[d] for d in dates]
        except KeyError as missing_date:
            raise ValueError(
                f"{nc_path}: date {missing_date} is not present on the "
                "file's time axis"
            ) from missing_date

        var = dataset.variables[variable]
        data = np.asarray(var[band_indices, :, :], dtype=np.float32)
        fill_value = getattr(var, "_FillValue", None)
        if fill_value is not None:
            data = np.where(data == fill_value, np.nan, data)

    if data.shape[1:] != shape:
        raise ValueError(
            f"{nc_path}: observation grid {data.shape[1:]} != model grid {shape}"
        )
    return data


def write_tif(array, profile, path):
    """Atomic (write-to-temp, then rename) single-band float32 GeoTIFF write."""
    import rasterio

    out_profile = dict(profile)
    out_profile.update(
        dtype="float32", count=1, compress="deflate", predictor=3, zlevel=4,
        nodata=np.nan, tiled=True, blockxsize=256, blockysize=256,
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(path.name + ".tmp")
    with rasterio.open(tmp_path, "w", **out_profile) as ds:
        ds.write(np.asarray(array, dtype=np.float32), 1)
    tmp_path.replace(path)


def write_year_multiband_tif(data, dates, profile, path, compress_level=4):
    """
    Write a whole year's corrected series as one multi-band GeoTIFF, in the
    same format as the yearly-aggregated *inputs* this notebook reads
    (band_dates JSON tag + per-band ISO-date descriptions), so it can be
    read back the same way via read_year_multiband_tif().

    data  : (len(dates), H, W) float32, chronological, matching `dates`
    dates : list of dt.date for axis 0 of `data`
    """
    import rasterio

    height, width = data.shape[1:]
    out_profile = dict(profile)
    out_profile.update(
        height=height, width=width, count=len(dates), dtype="float32",
        compress="deflate", predictor=3, zlevel=compress_level, nodata=np.nan,
        tiled=True, blockxsize=256, blockysize=256,
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(path.name + ".tmp")
    with rasterio.open(tmp_path, "w", **out_profile) as dst:
        dst.write(np.asarray(data, dtype=np.float32))
        dst.descriptions = tuple(d.isoformat() for d in dates)
        dst.update_tags(
            band_dates=jsonlib.dumps([d.isoformat() for d in dates]),
            year=str(dates[0].year) if dates else "",
            band_count=str(len(dates)),
        )
    tmp_path.replace(path)


def year_output_is_complete(path, expected_band_count):
    """True if `path` already exists and has exactly `expected_band_count`
    bands -- used to decide whether a yearly output file can be skipped."""
    import rasterio

    if not path.exists():
        return False
    try:
        with rasterio.open(path) as ds:
            return ds.count == expected_band_count
    except Exception:
        return False

## Synthetic self-test

This checks that fitting/current mapping/future QDM run and that finite corrected values are non-negative. It does not use the real files.

In [ ]:
def _selftest_qdm():
    rng = np.random.default_rng(123)

    fit_dates = date_range(
        dt.date(2015, 1, 1),
        dt.date(2023, 12, 31),
    )

    n = len(fit_dates)
    t = np.arange(
        n,
        dtype=np.float64,
    )

    seasonal = (
        0.4
        + 0.3
        * (
            1
            + np.sin(
                t * 2*np.pi / 365.25
            )
        )
    )

    obs1 = rng.gamma(
        shape=1.5,
        scale=seasonal,
    )

    mod1 = 0.75 * rng.gamma(
        shape=1.5,
        scale=seasonal * 1.2,
    )

    obs1[obs1 < 0.08] = 0.0
    mod1[mod1 < 0.08] = 0.0

    height = 2
    width = 3

    obs = np.tile(
        obs1[:, None, None],
        (1, height, width),
    ).astype(np.float32)

    mod = np.tile(
        mod1[:, None, None],
        (1, height, width),
    ).astype(np.float32)

    taus = np.linspace(
        TAU_MIN,
        TAU_MAX,
        N_QUANTILES,
    )

    oq, mq = fit_qdm_quantiles(
        obs,
        mod,
        fit_dates,
        taus,
        trace_calc=0.005,
        min_cal_days=30,
        seed_prefix="selftest_fit",
    )

    assert oq.shape == (
        12,
        N_QUANTILES,
        height,
        width,
    )

    assert mq.shape == (
        12,
        N_QUANTILES,
        height,
        width,
    )

    assert np.nanmin(oq) >= 0
    assert np.nanmin(mq) >= 0

    # Current-period example.
    dates2019 = date_range(
        dt.date(2019, 1, 1),
        dt.date(2019, 12, 31),
    )

    idx2019 = [
        i
        for i, d in enumerate(fit_dates)
        if d.year == 2019
    ]

    current = apply_current_qm(
        mod[idx2019],
        dates2019,
        oq,
        mq,
        taus,
        trace=0.01,
        trace_calc=0.005,
        seed_prefix="selftest_current",
        day_chunk=4,
    )

    assert current.shape == (
        365,
        height,
        width,
    )

    assert np.nanmin(current) >= 0

    # Future 9-year projection window.
    window = {}

    for yy in range(2024, 2033):
        dates = date_range(
            dt.date(yy, 1, 1),
            dt.date(yy, 12, 31),
        )

        tt = np.arange(
            len(dates),
            dtype=np.float64,
        )

        trend = (
            1.0
            + 0.03
            * (yy - 2024)
        )

        base = rng.gamma(
            shape=1.5,
            scale=(
                0.7
                + 0.3
                * np.sin(
                    tt * 2*np.pi / 365.25
                )
            )
            * trend,
        )

        base[
            base < 0.05
        ] = 0.0

        arr = np.tile(
            base[:, None, None],
            (1, height, width),
        ).astype(np.float32)

        window[yy] = (
            dates,
            arr,
        )

    target_dates, target_raw = window[
        2028
    ]

    corrected = apply_future_qdm_year(
        target_year=2028,
        target_dates=target_dates,
        target_raw=target_raw,
        window_year_data=window,
        obs_q=oq,
        mod_q=mq,
        taus=taus,
        trace=0.01,
        trace_calc=0.005,
        ratio_max=2.0,
        ratio_max_trace=0.05,
        seed_prefix="selftest_future",
        day_chunk=4,
    )

    assert corrected.shape == target_raw.shape
    assert np.nanmin(corrected) >= 0
    assert np.isfinite(corrected).any()

    print("QDM SELF-TEST PASSED")
    print("current min :", float(np.nanmin(current)))
    print("future min  :", float(np.nanmin(corrected)))
    print("future mean :", float(np.nanmean(corrected)))


_selftest_qdm()

## Stage 1 — fit monthly QDM quantiles

Each `(product, model, ssp)` stores monthly observation/model quantile curves in a compressed `.npz`.

In [ ]:
def qdm_param_path(
    product,
    model,
    ssp,
):
    return (
        PARAMS_DIR
        / product
        / model
        / f"{product}_{model}_{ssp}_qdm_params.npz"
    )


def fit_one(
    project_dir,
    product,
    model,
    ssp,
    fit_dates,
):
    out_path = qdm_param_path(
        product,
        model,
        ssp,
    )

    if (
        out_path.exists()
        and SKIP_EXISTING
    ):
        print(
            f"SKIP fit (exists): {out_path}"
        )
        return (
            product,
            model,
            ssp,
            "skipped",
        )

    yearly_dir = (
        project_dir
        / SSP_YEARLY_ROOT_REL.format(
            product=product
        )
        / ssp
        / model
    )

    obs_path = (
        project_dir
        / OBS_NC_REL.format(
            product=product
        )
    )

    obs_variable = OBS_VARIABLE.format(
        product=product
    )

    trace = float(
        TRACE_BY_PRODUCT[
            product
        ]
    )

    trace_calc = float(
        TRACE_CALC_BY_PRODUCT[
            product
        ]
    )

    taus = np.linspace(
        TAU_MIN,
        TAU_MAX,
        N_QUANTILES,
        dtype=np.float64,
    )

    print(
        f"[fit] {product}/{model}/{ssp}: "
        f"reading model {FIT_START}..{FIT_END}"
    )

    mod, reference_profile = (
        read_dates_from_yearly_dir(
            yearly_dir,
            fit_dates,
        )
    )

    shape = mod.shape[1:]

    print(
        f"[fit] {product}/{model}/{ssp}: "
        "reading observations"
    )

    obs = read_obs_series(
        obs_path,
        obs_variable,
        fit_dates,
        shape,
    )

    print(
        f"[fit] {product}/{model}/{ssp}: "
        "fitting monthly QDM quantiles"
    )

    obs_q, mod_q = fit_qdm_quantiles(
        obs,
        mod,
        fit_dates,
        taus,
        trace_calc=trace_calc,
        min_cal_days=MIN_CAL_DAYS_PER_MONTH,
        seed_prefix=(
            f"{product}|{model}|{ssp}"
        ),
    )

    finite_cells = int(
        (
            np.all(
                np.isfinite(obs_q),
                axis=(0, 1),
            )
            & np.all(
                np.isfinite(mod_q),
                axis=(0, 1),
            )
        ).sum()
    )

    total_cells = (
        shape[0]
        * shape[1]
    )

    print(
        f"[fit] {product}/{model}/{ssp}: "
        f"{finite_cells}/{total_cells} cells "
        "have complete quantile curves "
        f"({100*finite_cells/total_cells:.1f}%)"
    )

    out_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp_path = out_path.with_name(
        out_path.stem
        + ".tmp.npz"
    )

    ratio_max_trace = (
        RATIO_MAX_TRACE_FACTOR
        * max(
            trace,
            trace_calc,
        )
    )

    np.savez_compressed(
        tmp_path,
        obs_q=obs_q.astype(
            np.float32
        ),
        mod_q=mod_q.astype(
            np.float32
        ),
        taus=taus.astype(
            np.float32
        ),
        product=product,
        model=model,
        ssp=ssp,
        fit_start=fit_dates[
            0
        ].isoformat(),
        fit_end=fit_dates[
            -1
        ].isoformat(),
        trace=np.float32(
            trace
        ),
        trace_calc=np.float32(
            trace_calc
        ),
        ratio_max=np.float32(
            RATIO_MAX
        ),
        ratio_max_trace=np.float32(
            ratio_max_trace
        ),
        qdm_window_years=np.int32(
            QDM_WINDOW_YEARS
        ),
        crs_wkt=reference_profile[
            "crs"
        ].to_wkt(),
        transform_gdal=np.asarray(
            reference_profile[
                "transform"
            ].to_gdal()
        ),
        height=np.int32(
            shape[0]
        ),
        width=np.int32(
            shape[1]
        ),
    )

    tmp_path.replace(
        out_path
    )

    print(
        f"[fit] wrote {out_path}"
    )

    del (
        obs,
        mod,
        obs_q,
        mod_q,
    )

    gc.collect()

    return (
        product,
        model,
        ssp,
        "written",
    )

In [ ]:
fit_dates = date_range(
    FIT_START,
    FIT_END,
)

fit_tasks = [
    (
        product,
        model,
        ssp,
    )
    for product in PRODUCTS
    for model in MODELS
    for ssp in SSPS
]

print(
    f"Fit window: {FIT_START} .. {FIT_END} "
    f"({len(fit_dates)} days)"
)

print(
    f"{len(fit_tasks)} fit task(s); "
    f"FIT_PROCESSES={FIT_PROCESSES}"
)

fit_failures = []

if FIT_PROCESSES == 1:
    for (
        product,
        model,
        ssp,
    ) in fit_tasks:
        try:
            fit_one(
                PROJECT_DIR,
                product,
                model,
                ssp,
                fit_dates,
            )
        except Exception:
            print(
                f"ERROR fit "
                f"[{product}/{model}/{ssp}]:",
                file=sys.stderr,
            )
            traceback.print_exc()
            fit_failures.append(
                (
                    product,
                    model,
                    ssp,
                )
            )

else:
    with ProcessPoolExecutor(
        max_workers=FIT_PROCESSES
    ) as executor:

        futures = {
            executor.submit(
                fit_one,
                PROJECT_DIR,
                product,
                model,
                ssp,
                fit_dates,
            ): (
                product,
                model,
                ssp,
            )
            for (
                product,
                model,
                ssp,
            ) in fit_tasks
        }

        for future in as_completed(
            futures
        ):
            (
                product,
                model,
                ssp,
            ) = futures[
                future
            ]

            try:
                future.result()

            except Exception:
                print(
                    f"ERROR fit "
                    f"[{product}/{model}/{ssp}]:",
                    file=sys.stderr,
                )
                traceback.print_exc()
                fit_failures.append(
                    (
                        product,
                        model,
                        ssp,
                    )
                )

if fit_failures:
    print(
        f"\n{len(fit_failures)} "
        "fit task(s) failed:"
    )

    for item in fit_failures:
        print(
            " ",
            item,
        )

else:
    print(
        "\nAll QDM fit tasks completed."
    )

## Stage 2 — apply QDM

The future step uses a moving multi-year cache, so QDM is more memory-intensive than your Hempel notebook. Start with `APPLY_PROCESSES = 4`.

In [ ]:
def load_qdm_params(
    product,
    model,
    ssp,
):
    from affine import Affine
    from rasterio.crs import CRS

    path = qdm_param_path(
        product,
        model,
        ssp,
    )

    with np.load(
        path,
        allow_pickle=False,
    ) as npz:

        obs_q = npz[
            "obs_q"
        ]

        mod_q = npz[
            "mod_q"
        ]

        taus = np.asarray(
            npz["taus"],
            dtype=np.float64,
        )

        height = int(
            npz["height"]
        )

        width = int(
            npz["width"]
        )

        trace = float(
            npz["trace"]
        )

        trace_calc = float(
            npz["trace_calc"]
        )

        ratio_max = float(
            npz["ratio_max"]
        )

        ratio_max_trace = float(
            npz[
                "ratio_max_trace"
            ]
        )

        crs = CRS.from_wkt(
            str(
                npz[
                    "crs_wkt"
                ]
            )
        )

        transform = Affine.from_gdal(
            *npz[
                "transform_gdal"
            ].tolist()
        )

    profile = {
        "driver": "GTiff",
        "height": height,
        "width": width,
        "count": 1,
        "crs": crs,
        "transform": transform,
    }

    return {
        "obs_q": obs_q,
        "mod_q": mod_q,
        "taus": taus,
        "shape": (
            height,
            width,
        ),
        "trace": trace,
        "trace_calc": trace_calc,
        "ratio_max": ratio_max,
        "ratio_max_trace": ratio_max_trace,
        "profile": profile,
    }


def _load_year_into_cache(
    cache,
    yearly_dir,
    year,
    expected_shape,
):
    if year not in cache:
        path = (
            yearly_dir
            / f"{year}.tif"
        )

        dates, data, profile = (
            read_year_multiband_tif(
                path
            )
        )

        if data.shape[1:] != expected_shape:
            raise ValueError(
                f"{path}: grid "
                f"{data.shape[1:]} != "
                f"parameter grid "
                f"{expected_shape}"
            )

        cache[year] = (
            dates,
            data,
        )

    return cache[
        year
    ]


def _trim_cache(
    cache,
    keep_years,
):
    keep = set(
        keep_years
    )

    for yy in list(
        cache
    ):
        if yy not in keep:
            del cache[
                yy
            ]

    gc.collect()


def apply_one_task(
    project_dir,
    product,
    model,
    ssp,
    apply_start,
    apply_end,
    skip_existing,
    output_mode,
):
    p = load_qdm_params(
        product,
        model,
        ssp,
    )

    obs_q = p[
        "obs_q"
    ]
    mod_q = p[
        "mod_q"
    ]
    taus = p[
        "taus"
    ]
    shape = p[
        "shape"
    ]
    trace = p[
        "trace"
    ]
    trace_calc = p[
        "trace_calc"
    ]
    ratio_max = p[
        "ratio_max"
    ]
    ratio_max_trace = p[
        "ratio_max_trace"
    ]
    profile = p[
        "profile"
    ]

    yearly_dir = (
        project_dir
        / SSP_YEARLY_ROOT_REL.format(
            product=product
        )
        / ssp
        / model
    )

    cache = {}

    written_years = 0
    written_days = 0
    skipped_years = 0

    future_first_year = (
        QDM_PROJECTION_START.year
    )

    future_last_year = (
        apply_end.year
    )

    for year in range(
        apply_start.year,
        apply_end.year + 1,
    ):

        year_start = max(
            apply_start,
            dt.date(
                year,
                1,
                1,
            ),
        )

        year_end = min(
            apply_end,
            dt.date(
                year,
                12,
                31,
            ),
        )

        expected_dates = date_range(
            year_start,
            year_end,
        )

        if output_mode == "yearly":
            year_out_path = (
                OUTPUT_DIR
                / product
                / ssp
                / model
                / f"{year}.tif"
            )

            if (
                skip_existing
                and year_output_is_complete(
                    year_out_path,
                    len(
                        expected_dates
                    ),
                )
            ):
                skipped_years += 1
                continue

        else:
            day_out_paths = [
                (
                    OUTPUT_DIR
                    / product
                    / ssp
                    / model
                    / str(year)
                    / f"{d.isoformat()}.tif"
                )
                for d in expected_dates
            ]

            if (
                skip_existing
                and all(
                    path.exists()
                    for path
                    in day_out_paths
                )
            ):
                skipped_years += 1
                continue

        # ====================================================
        # Current/calibration-period mapping
        # ====================================================

        if year < future_first_year:

            (
                full_dates,
                full_raw,
            ) = _load_year_into_cache(
                cache,
                yearly_dir,
                year,
                shape,
            )

            date_to_band = {
                d: i
                for i, d
                in enumerate(
                    full_dates
                )
            }

            indices = [
                date_to_band[d]
                for d
                in expected_dates
            ]

            raw = full_raw[
                indices
            ]

            corrected = apply_current_qm(
                raw=raw,
                dates=expected_dates,
                obs_q=obs_q,
                mod_q=mod_q,
                taus=taus,
                trace=trace,
                trace_calc=trace_calc,
                seed_prefix=(
                    f"{product}|"
                    f"{model}|"
                    f"{ssp}|"
                    f"{year}"
                ),
                day_chunk=DAY_CHUNK,
            )

            _trim_cache(
                cache,
                [year],
            )

        # ====================================================
        # Future ratio-QDM
        # ====================================================

        else:

            window_years = (
                projection_window_years(
                    target_year=year,
                    first_year=(
                        future_first_year
                    ),
                    last_year=(
                        future_last_year
                    ),
                    width=(
                        QDM_WINDOW_YEARS
                    ),
                )
            )

            for yy in window_years:
                _load_year_into_cache(
                    cache,
                    yearly_dir,
                    yy,
                    shape,
                )

            (
                full_target_dates,
                full_target_raw,
            ) = cache[
                year
            ]

            date_to_band = {
                d: i
                for i, d
                in enumerate(
                    full_target_dates
                )
            }

            indices = [
                date_to_band[d]
                for d
                in expected_dates
            ]

            target_raw = (
                full_target_raw[
                    indices
                ]
            )

            corrected = (
                apply_future_qdm_year(
                    target_year=year,
                    target_dates=(
                        expected_dates
                    ),
                    target_raw=(
                        target_raw
                    ),
                    window_year_data=(
                        cache
                    ),
                    obs_q=obs_q,
                    mod_q=mod_q,
                    taus=taus,
                    trace=trace,
                    trace_calc=(
                        trace_calc
                    ),
                    ratio_max=(
                        ratio_max
                    ),
                    ratio_max_trace=(
                        ratio_max_trace
                    ),
                    seed_prefix=(
                        f"{product}|"
                        f"{model}|"
                        f"{ssp}"
                    ),
                    day_chunk=(
                        DAY_CHUNK
                    ),
                )
            )

            _trim_cache(
                cache,
                window_years,
            )

        # Final safety.
        finite = np.isfinite(
            corrected
        )

        corrected[
            finite
        ] = np.maximum(
            corrected[
                finite
            ],
            0.0,
        )

        if output_mode == "yearly":
            write_year_multiband_tif(
                corrected,
                expected_dates,
                profile,
                year_out_path,
            )

            written_years += 1

        else:
            for (
                day_index,
                out_path,
            ) in enumerate(
                day_out_paths
            ):

                if (
                    skip_existing
                    and out_path.exists()
                ):
                    continue

                write_tif(
                    corrected[
                        day_index
                    ],
                    profile,
                    out_path,
                )

                written_days += 1

        print(
            f"[apply] "
            f"{product}/"
            f"{model}/"
            f"{ssp}/"
            f"{year}: "
            f"min="
            f"{np.nanmin(corrected):.6g}, "
            f"mean="
            f"{np.nanmean(corrected):.6g}, "
            f"negative="
            f"{int(np.sum(corrected < 0))}"
        )

        del corrected
        gc.collect()

    print(
        f"[apply] DONE "
        f"{product}/"
        f"{model}/"
        f"{ssp}: "
        f"wrote {written_years} "
        "yearly file(s) / "
        f"{written_days} daily file(s); "
        f"skipped {skipped_years} "
        "complete year(s)"
    )

    return (
        product,
        model,
        ssp,
        written_years,
        written_days,
        skipped_years,
    )

In [ ]:
apply_tasks = [
    (
        product,
        model,
        ssp,
    )
    for product in PRODUCTS
    for model in MODELS
    for ssp in SSPS
]

print(
    f"Apply window: "
    f"{APPLY_START} .. "
    f"{APPLY_END}"
)

print(
    f"QDM future starts: "
    f"{QDM_PROJECTION_START}"
)

print(
    f"{len(apply_tasks)} "
    "product/model/ssp task(s); "
    f"{APPLY_PROCESSES} "
    "parallel process(es)"
)

apply_failures = []

with ProcessPoolExecutor(
    max_workers=APPLY_PROCESSES
) as executor:

    futures = {
        executor.submit(
            apply_one_task,
            PROJECT_DIR,
            product,
            model,
            ssp,
            APPLY_START,
            APPLY_END,
            SKIP_EXISTING,
            OUTPUT_MODE,
        ): (
            product,
            model,
            ssp,
        )
        for (
            product,
            model,
            ssp,
        ) in apply_tasks
    }

    for future in as_completed(
        futures
    ):
        (
            product,
            model,
            ssp,
        ) = futures[
            future
        ]

        try:
            future.result()

        except Exception:
            print(
                f"ERROR apply "
                f"[{product}/"
                f"{model}/"
                f"{ssp}]:",
                file=sys.stderr,
            )

            traceback.print_exc()

            apply_failures.append(
                (
                    product,
                    model,
                    ssp,
                )
            )

if apply_failures:
    print(
        f"\n{len(apply_failures)} "
        "apply task(s) failed:"
    )

    for (
        product,
        model,
        ssp,
    ) in apply_failures:

        print(
            f"  {product}/"
            f"{model}/"
            f"{ssp}"
        )

else:
    print(
        "\nAll QDM apply tasks completed."
    )

## Quick diagnostic

Run after a smoke test. The function reports yearly min/mean/max and number of negative pixels.

In [ ]:
def diagnose_outputs(
    product,
    model,
    ssp,
    years_to_check,
):
    rows = []

    for year in years_to_check:
        path = (
            OUTPUT_DIR
            / product
            / ssp
            / model
            / f"{year}.tif"
        )

        if not path.exists():
            rows.append(
                (
                    year,
                    np.nan,
                    np.nan,
                    np.nan,
                    -1,
                )
            )
            continue

        dates, data, _ = (
            read_year_multiband_tif(
                path
            )
        )

        finite = np.isfinite(
            data
        )

        rows.append(
            (
                year,
                float(
                    np.nanmin(
                        data
                    )
                ),
                float(
                    np.nanmean(
                        data
                    )
                ),
                float(
                    np.nanmax(
                        data
                    )
                ),
                int(
                    np.sum(
                        finite
                        & (
                            data
                            < 0
                        )
                    )
                ),
            )
        )

    print(
        "year       min          "
        "mean          max       "
        "negative"
    )

    for row in rows:
        print(
            f"{row[0]:4d}  "
            f"{row[1]:11.5g}  "
            f"{row[2]:11.5g}  "
            f"{row[3]:11.5g}  "
            f"{row[4]:9d}"
        )

    return rows


# Example:
#
# diagnose_outputs(
#     product="firepm25",
#     model="ACCESS-CM2",
#     ssp="ssp126",
#     years_to_check=[
#         2015,
#         2020,
#         2023,
#         2024,
#         2040,
#         2080,
#         2100,
#     ],
# )

## Recommended sensitivity checks

Before adopting QDM as the main correction, compare:

- `QDM_WINDOW_YEARS = 5, 9, 15`
- `N_QUANTILES = 21, 41, 81`
- `RATIO_MAX = 2, 5`
- alternative `TRACE` values appropriate for the pollutant units

For your current result, specifically compare the **2040s and 2080s peaks** under:
1. raw model,
2. Hempel,
3. QDM,

and inspect both the 17-model ensemble **mean and median**.